# ANAADHI — P1 Root Location Factory

GitHub-managed, Android/Kaggle-friendly environment generator.

**Current default job: `LOC-016` — Forest-Edge Barter Hamlet correction.**

This notebook generates **four clean environment-identity candidates** for one canonical LOC at a time. It does **not** generate storyboard action, characters, costumes, props, dialogue, or camera grammar.

Use **Tesla T4** and keep Kaggle Internet ON.


In [ ]:
import sys, subprocess, importlib.util

required = ["diffusers", "transformers", "accelerate", "safetensors", "peft"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("Required AI packages already available — skipping install.")

import os, json, urllib.request, math, torch
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from diffusers import AutoPipelineForText2Image
from IPython.display import display

assert torch.cuda.is_available(), "GPU is not enabled. Kaggle Settings → Accelerator → GPU T4."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("CUDA:", torch.version.cuda)
if "P100" in GPU_NAME.upper():
    raise RuntimeError("P100 is not supported by this ANAADHI notebook. Select Tesla T4.")


## Canonical job manifest

The notebook reads the current Wave-A correction manifest directly from GitHub. The manifest is subordinate to the frozen ENV-6J/ENV-6L authority.


In [ ]:
MANIFEST_URL = "https://raw.githubusercontent.com/amoghavarshavpatil-dot/anaadhi-storyboard-generator/main/production/locations/ROOT_FACTORY_WAVE_A.json"
with urllib.request.urlopen(MANIFEST_URL, timeout=30) as r:
    ROOT_MANIFEST = json.loads(r.read().decode("utf-8"))

print("Phase:", ROOT_MANIFEST["phase"])
print("Current jobs:", [j["loc_id"] for j in ROOT_MANIFEST["current_wave"]])
print("Next roots after correction approval:", ROOT_MANIFEST["next_wave_after_approval"][0], "→", ROOT_MANIFEST["next_wave_after_approval"][-1])


## Current job

Do not edit prompts on the phone. ChatGPT will change the GitHub manifest after review. The default remains LOC-016 until one of its four candidates is approved.


In [ ]:
JOB_ID = "LOC-016"

jobs = {j["loc_id"]: j for j in ROOT_MANIFEST["current_wave"]}
if JOB_ID not in jobs:
    raise KeyError(f"{JOB_ID} is not in the current GitHub correction wave.")

job = jobs[JOB_ID]
PROMPT = job["sdxl_prompt"]
NEGATIVE_PROMPT = job["negative_prompt"]
SEEDS = job["seeds"]

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
STEPS = 32
GUIDANCE = 6.5
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608

OUTPUT_DIR = Path("/kaggle/working/anaadhi_location_roots") / JOB_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("LOC:", JOB_ID)
print("Canonical name:", job["canonical_name"])
print("Task:", job["task"])
print("Canonical identity:", job["canonical_identity"])
print("Seeds:", SEEDS)


## Load SDXL once

The first model load is the slow part. All four candidates reuse the same loaded model.


In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_slicing()
except Exception:
    pass

print("Model ready:", MODEL_ID)

def token_count(tokenizer, text):
    encoded = tokenizer(text, truncation=False, add_special_tokens=True)
    return len(encoded.input_ids)

for label, text in [("positive", PROMPT), ("negative", NEGATIVE_PROMPT)]:
    c1 = token_count(pipe.tokenizer, text)
    c2 = token_count(pipe.tokenizer_2, text) if getattr(pipe, "tokenizer_2", None) else c1
    max1 = getattr(pipe.tokenizer, "model_max_length", 77)
    max2 = getattr(getattr(pipe, "tokenizer_2", None), "model_max_length", max1) if getattr(pipe, "tokenizer_2", None) else max1
    print(f"{label} tokens: CLIP1={c1}/{max1}, CLIP2={c2}/{max2}")
    if c1 > max1 or c2 > max2:
        raise RuntimeError(f"{label} prompt exceeds CLIP limit. Stop and let ChatGPT compact the GitHub manifest.")


## Generate four candidates

Each candidate is cropped to the same practical scope ratio as the 3840×1608 ANAADHI master. Candidate PNGs contain **no baked labels or black bars**. A separate contact sheet is created only for easy phone review.


In [ ]:
target_ratio = MASTER_WIDTH / MASTER_HEIGHT
saved = []
preview_images = []

def scope_crop(image):
    new_h = round(image.width / target_ratio)
    if new_h <= image.height:
        y0 = (image.height - new_h) // 2
        return image.crop((0, y0, image.width, y0 + new_h))
    new_w = round(image.height * target_ratio)
    x0 = (image.width - new_w) // 2
    return image.crop((x0, 0, x0 + new_w, image.height))

for seed in SEEDS:
    print(f"Generating {JOB_ID} candidate seed {seed}...")
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=GEN_WIDTH,
        height=GEN_HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]

    scoped = scope_crop(image)
    out_path = OUTPUT_DIR / f"{JOB_ID}_ST00_seed{seed}.png"
    scoped.save(out_path)
    saved.append(str(out_path))
    preview_images.append((seed, scoped.copy()))
    print("Saved:", out_path, "size:", scoped.size, "ratio:", round(scoped.width/scoped.height, 4))
    display(scoped)

thumb_w = 672
thumb_h = round(thumb_w / target_ratio)
label_h = 34
sheet = Image.new("RGB", (thumb_w*2, (thumb_h+label_h)*2), "white")
draw = ImageDraw.Draw(sheet)

for idx, (seed, img) in enumerate(preview_images):
    thumb = img.resize((thumb_w, thumb_h))
    x = (idx % 2) * thumb_w
    y = (idx // 2) * (thumb_h + label_h)
    sheet.paste(thumb, (x, y))
    draw.text((x+10, y+thumb_h+8), f"{JOB_ID} — seed {seed}", fill="black")

sheet_path = OUTPUT_DIR / f"{JOB_ID}_CONTACT_SHEET.jpg"
sheet.save(sheet_path, quality=92)

print("\nFinished", len(saved), "candidates.")
print("Candidate folder:", OUTPUT_DIR)
print("Contact sheet:", sheet_path)
display(sheet)


## Review gate

Send the four candidates/contact sheet to ChatGPT.

- **No file is automatically approved.**
- If none match canon, ChatGPT updates GitHub and the job is regenerated.
- If one passes, its seed/file becomes the approved root candidate and GitHub advances the default job.
- LOC-040 follows LOC-016.
- After both correction jobs pass, the factory moves to LOC-060–LOC-087.
